In [2]:
from kiwipiepy import Kiwi
kiwi = Kiwi()

In [3]:
kiwi.tokenize("안녕하세요 형태소 분석기 키위입니다.")

[Token(form='안녕', tag='NNG', start=0, len=2),
 Token(form='하', tag='XSA', start=2, len=1),
 Token(form='세', tag='EC', start=3, len=1),
 Token(form='요', tag='JX', start=4, len=1),
 Token(form='형태소', tag='NNG', start=6, len=3),
 Token(form='분석기', tag='NNG', start=10, len=3),
 Token(form='키위', tag='NNG', start=14, len=2),
 Token(form='이', tag='VCP', start=16, len=1),
 Token(form='ᆸ니다', tag='EF', start=16, len=3),
 Token(form='.', tag='SF', start=19, len=1)]

In [10]:
import jsonlines
from kiwipiepy import Kiwi
from rank_bm25 import BM25Okapi

# Kiwi 토크나이저 생성
kiwi = Kiwi()

# JSONL 파일 경로
jsonl_file_path = '/upstage-ai-advanced-ir7/data/documents.jsonl'

stoptags = {"E", "J", "SC", "SE", "SF", "SP", "VX", "VCN", "VCP"}

# 문서와 docid 저장할 리스트
documents = []
docids = []

# JSONL 파일 읽기
with jsonlines.open(jsonl_file_path) as reader:
    for obj in reader:
        docids.append(obj['docid'])      # docid 저장
        documents.append(obj['content']) # content 저장

# Kiwi를 사용하여 문서 토큰화
def tokenize_with_kiwi(text):
    tokens = kiwi.tokenize(text)  # Kiwi 결과를 토큰화
    processed_tokens = []
    
    for token in tokens:
        word = token.form          # 형태소 단어
        pos_tag = token.tag        # 품사 태그
        # if pos_tag not in stoptags:  # 불필요한 품사 태그가 아닌 경우에만 추가
        processed_tokens.append(word)
    
    return processed_tokens

# 각 문서를 Kiwi로 토큰화
tokenized_corpus = [tokenize_with_kiwi(doc) for doc in documents]

# BM25 인덱서 생성
bm25 = BM25Okapi(tokenized_corpus)


KeyboardInterrupt: 

In [22]:
stoptags = {"E", "J", "SC", "SE", "SF", "SP", "VX", "VCN", "VCP"}

In [25]:
def tokenize_with_kiwi(text):
    tokens = kiwi.tokenize(text)  # Kiwi 결과를 토큰화
    processed_tokens = []
    
    for token in tokens:
        word = token.form          # 형태소 단어
        pos_tag = token.tag        # 품사 태그
        if pos_tag not in stoptags:  # 불필요한 품사 태그가 아닌 경우에만 추가
            processed_tokens.append(word)
    
    return processed_tokens

In [26]:
query = "건강한 사람이 에너지 균형을 평형 상태로 유지하는 것은 중요합니다. 에너지 균형은 에너지 섭취와 에너지 소비의"  # 예시 쿼리
tokenized_query = tokenize_with_kiwi(query)
tokenized_query

['건강',
 '하',
 'ᆫ',
 '사람',
 '이',
 '에너지',
 '균형',
 '을',
 '평형',
 '상태',
 '로',
 '유지',
 '하',
 '는',
 '것',
 '은',
 '중요',
 '하',
 'ᆸ니다',
 '에너지',
 '균형',
 '은',
 '에너지',
 '섭취',
 '와',
 '에너지',
 '소비',
 '의']

In [24]:
query = "건강한 사람이 에너지 균형을 평형 상태로 유지하는 것은 중요합니다. 에너지 균형은 에너지 섭취와 에너지 소비의"  # 예시 쿼리
tokenized_query = tokenize_with_kiwi(query)
tokenized_query

[Token(form='건강', tag='NNG', start=0, len=2),
 Token(form='하', tag='XSA', start=2, len=1),
 Token(form='ᆫ', tag='ETM', start=2, len=1),
 Token(form='사람', tag='NNG', start=4, len=2),
 Token(form='이', tag='JKS', start=6, len=1),
 Token(form='에너지', tag='NNG', start=8, len=3),
 Token(form='균형', tag='NNG', start=12, len=2),
 Token(form='을', tag='JKO', start=14, len=1),
 Token(form='평형', tag='NNG', start=16, len=2),
 Token(form='상태', tag='NNG', start=19, len=2),
 Token(form='로', tag='JKB', start=21, len=1),
 Token(form='유지', tag='NNG', start=23, len=2),
 Token(form='하', tag='XSV', start=25, len=1),
 Token(form='는', tag='ETM', start=26, len=1),
 Token(form='것', tag='NNB', start=28, len=1),
 Token(form='은', tag='JX', start=29, len=1),
 Token(form='중요', tag='NNG', start=31, len=2),
 Token(form='하', tag='XSA', start=33, len=1),
 Token(form='ᆸ니다', tag='EF', start=33, len=3),
 Token(form='.', tag='SF', start=36, len=1),
 Token(form='에너지', tag='NNG', start=38, len=3),
 Token(form='균형', tag='NNG', s

In [13]:
tokenized_query

In [6]:
query = "금성에서 달의 관측 모습"  # 예시 쿼리
tokenized_query = tokenize_with_kiwi(query)

# BM25로 점수 계산
doc_scores = bm25.get_scores(tokenized_query)

# 점수가 높은 순서대로 문서 정렬 (상위 200개만)
ranked_docs = sorted(zip(docids, doc_scores), key=lambda x: x[1], reverse=True)[:200]

# 결과 출력 (상위 200개만)
print("검색 결과 (상위 200개):")
for docid, score in ranked_docs:
    print(f"DocID: {docid}, Score: {score}")

검색 결과 (상위 200개):
DocID: 35c5dcc7-4720-4318-901e-770105ae63fd, Score: 23.239141705078485
DocID: 553989d9-ee23-4203-b244-a941b6fa8d99, Score: 21.73537844896049
DocID: b2e0e809-c9e9-4465-9248-07a9b49b034f, Score: 20.647831692209124
DocID: efb313ef-d7af-4d82-86f4-b5f013714a0c, Score: 19.90247007910991
DocID: bb6d04b6-a6cf-4a9f-8324-4e06e6e81c86, Score: 17.715659461159014
DocID: 45b8eb6a-87e3-4333-b01b-7c8b772f827f, Score: 17.610218583802073
DocID: da6c8a3f-45a9-4025-a63a-47c05ba2b336, Score: 17.30311599572166
DocID: 340485f8-4e78-44f4-a53a-2df21915367f, Score: 16.923758558273242
DocID: 8a78364e-63bf-4915-b718-fdc461bc62c9, Score: 16.72229969734233
DocID: 2b40e339-174c-462f-8607-7a6be35ccd6e, Score: 16.582890663029787
DocID: d1cbb6a8-6346-4e84-b294-0c8e84d37c07, Score: 16.526801414845174
DocID: 43b53301-468b-41a2-ad67-63d8ecd84596, Score: 16.393394466832742
DocID: 464ace62-ddf2-423d-a5d7-2f17e6785c8e, Score: 16.326008259321092
DocID: 79216c43-fe13-4413-abcc-a8b9f70dcdad, Score: 16.090037190